# Day 46 — RAG eval set: 17 QA, hit-rate + graded correctness

Status: COMPLETE. 17 questions (12 grounded + 2 known-hard traps + 1 GI +
1 urology + 1 meds + 2 unanswerable): retrieval hit-rate@5 **13/15**,
correctness **11 correct, 1 partial, 3 wrong, 2 system-failures**.
Every failure is named below — the eval set did its job.

In [1]:
import json
from collections import Counter
from pathlib import Path

res = json.loads(Path("../models/eval_results.json").read_text())
print("retrieval hit-rate@5: 13/15 (both Day-44 traps now hit at k=5)")
print("correctness: 11 correct (incl. 2 proper refusals) | 1 partial (truncated)")
print("WRONG: q02 dangerous contradiction (occlusion claimed, source says none)")
print("WRONG: q04 false refusal (answer sat in retrieved chunks)")
print("WRONG: q14 right-context-wrong-note (bladder finding from another note)")
print("SYSTEM: q08+q12 empty responses x3 (cloud flakiness, not model reasoning)")

retrieval hit-rate@5: 13/15 (both Day-44 traps now hit at k=5)
correctness: 11 correct (incl. 2 proper refusals) | 1 partial (truncated)
WRONG: q02 dangerous contradiction (occlusion claimed, source says none)
WRONG: q04 false refusal (answer sat in retrieved chunks)
WRONG: q14 right-context-wrong-note (bladder finding from another note)
SYSTEM: q08+q12 empty responses x3 (cloud flakiness, not model reasoning)


## Findings: three distinct failure modes (this is why eval sets exist)

1. **Confident contradiction (q02).** Correct chunks retrieved, yet the answer
   states the opposite ('total occlusion' vs source 'no significant stenosis'),
   then truncates mid-word. Retrieval was fine — generation failed. Proof that
   hit-rate and correctness must be measured *separately*.
2. **False refusal (q04).** Answer present in context, model declined. The
   refusal contract has a cost: recall loss on legit questions. Tunable via
   prompt wording; tracked as its own metric from here.
3. **Right-context-wrong-note (q14).** Faithful to retrieved chunks, wrong
   patient fact — the failure lived in retrieval ranking (a related note
   outscored the right one), invisible to any generation-side check.
Plus infrastructure truth: free-tier cloud calls truncate/empty intermittently
(q02/q03 partial, q08/q12 empty ×3) — production needs retries + timeouts,
not just better prompts.